# 05. Enums

**Tier:** Intermediate. **Why you'd use this:** A UI theme should be `"light"` or `"dark"`, not any free string. An enum makes the allowed members explicit and singleton.

**After this lesson you can:** define an `Enum`; compare members by identity; look up a member by value.
About 25 minutes. **Uses:** [Classes](01-classes-lesson.ipynb)


## Quick review

1. What is a property under the hood?
2. What does a classmethod receive?

<details><summary>Answers</summary>

A descriptor. The class.
</details>


## The idea

An `Enum` defines a closed set of named members. Each member is a singleton. Look up by value with `Color("red")`. Prefer members over bare strings when the set of choices is fixed.

<details><summary>Explain more</summary>

`Color.RED is Color("red")` is true. Unknown values raise instead of passing a typo downstream.
</details>


## See it work

Map a color name to an enum value.


In [1]:
from enum import Enum


class Color(Enum):
    RED = "red"
    BLUE = "blue"


def color_of(name):
    # 1. Members are singletons looked up by value.
    return Color(name).value


print(color_of("red"))
print(Color.RED is Color("red"))

red
True


## Check yourself

Answer these before you run the cell.

1. Are two lookups of `Color.RED` the same object?
2. What happens if you call `Color("green")`?


In [2]:
from enum import Enum


class Color(Enum):
    RED = "red"
    BLUE = "blue"


print(Color.RED is Color.RED)
print(Color.RED is Color("red"))

True
True


<details><summary>Answers</summary>

Yes. Looking up an unknown value raises `ValueError`.
</details>


## Why it works this way

Enum members are created at class definition time and become singletons. `Color.RED is Color.RED` stays true for the life of the process. Lookup by value with `Color("red")` returns that same member. That identity stability is why enums beat raw strings for closed sets of choices: comparisons can use `is`, typos fail early, and exhaustiveness is easier to see. Deep dive: [Names and objects](../../internals/01-objects/01-names-lesson.ipynb).

Under the hood an enum class still uses ordinary class machinery, with a metaclass that builds the members and prevents normal instantiation from creating new ones. The `.name` and `.value` attributes are the member's spelling and payload. Iteration walks members in definition order.

Cost is a small set of objects created once. The design reason is a typed vocabulary for states and modes. Prefer enums when the set is closed and meaningful. Prefer strings when the set is open or comes from uncontrolled external text you only pass through.

Evidence: `Color.RED is Color("red")`, and building an unknown value raises. Using a bare string in a branch silently accepts typos that an enum would reject at the boundary.


Enums also improve readability at call sites. `activate(Color.RED)` says more than `activate("red")`, and tools can warn on missing cases more easily. Keep values simple, usually strings or ints that serialize cleanly. Avoid putting mutable objects in enum values. When an external API sends strings, convert once at the edge with `Color(name)` and keep members elsewhere. That boundary conversion is where `ValueError` belongs.


In [3]:
from enum import Enum


class Color(Enum):
    RED = "red"
    BLUE = "blue"


print(Color.RED is Color("red"))
print(Color.RED.value)
print(Color.RED.name)
print(list(Color))

True
red
RED
[<Color.RED: 'red'>, <Color.BLUE: 'blue'>]


In [4]:
from enum import Enum


class Color(Enum):
    RED = "red"
    BLUE = "blue"


Color("green")

ValueError: 'green' is not a valid Color

## When to use it

Use an enum for a closed set of named choices. Use strings when values are open-ended external text. Compare members with `is` when you hold members, or compare `.value` when you are at a string boundary. Don't silently accept arbitrary strings for fixed modes. If a string crosses a process boundary, convert it back to a member as soon as it re-enters your code.


## Common mistake

Using a bare string through the stack and only failing when a typo hits a distant branch.


In [5]:
def paint(mode):
    if mode == "reed":
        return "red"
    if mode == "blue":
        return "blue"
    return "unknown"


print(paint("red"))

unknown


Parse into an enum at the boundary, then pass members inward.


In [6]:
from enum import Enum


class Color(Enum):
    RED = "red"
    BLUE = "blue"


def paint(mode):
    color = Color(mode)
    return color.value


print(paint("red"))

red


## Try it

Add `GREEN = "green"` and write a function that returns the member for a name, not the string value. How do callers compare the result?


## Recap

- Enum members are singletons.
- Lookup by value returns the member or raises.
- Prefer enums for closed choice sets.

**Next:** [practice](05-enums-practice.ipynb)
